In [1]:
"""
Advanced cost model for LPG chain (uniform cost allocation, 2026 update):
1) Filling outbound cost: fixed cost per kg based on reference capacity (35,000 ton/year, Gurkan et al. 2026), independent of actual demand.
2) Reseller inbound cost: fixed cost per kg based on system-average reseller demand (excluding direct filling clients).
3) Truck transport: uses actual one-way distances (filling_ref_distance_km) for variable cost; fixed cost per kg based on reference annual truck utilization (5,000 km/year) and average round-trip distance.

Key assumptions:
- All fixed costs are allocated using reference or average demand, not actual demand per point.
- Actual path distances are used for transport costs (no time-based distance estimates).
- End-user collection cost is calculated per trip (per cylinder), not annualized.
- All costs in USD/kg; distances in km; times in hours; VOT in USD/h.
- Source: Gurkan et al. (2026) and local assumptions (see inline comments).
"""

from __future__ import annotations

from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd

# =============================================================================
# PATHS AND LAYERS
# =============================================================================
DATA_DIR = Path("dataset_big")
CHAIN_GPKG = DATA_DIR / "chain_with_cost.gpkg"
SELL_POINT_GPKG = DATA_DIR / "sell_point_clients.gpkg"
FILLING_POINT_GPKG = DATA_DIR / "filling_point_clients.gpkg"

RESELL_LAYER = "resell"
FILLING_LAYER = "filling"
SELL_POINT_LAYER = "sell_point_clients"
FILLING_POINT_LAYER = "filling_point_clients"

# =============================================================================
# KEYS AND COLUMNS
# =============================================================================
ID_COL = "id_res&fil"
FILL_REF_COL = "filling_reference"
FILL_TIME_COL = "filling_ref_time_min"
FILL_COST_IN_COL = "cost_fil_kg_in"
FILL_COST_OUT_COL = "cost_fil_kg_out"
RESELL_COST_IN_COL = "cost_res_kg_in"
FILL_ORIGIN_COST_OUT_COL = "cost_fil_kg_out_reference"
SELL_CLIENTS_COL = "clients"
FILL_TOTAL_CLIENTS_COL = "total_fil_clients"
FILL_ASSIGNED_CLIENTS_COL = "assigned_fil_clients"
FILL_ASSIGNED_MAX_IDEAL_CLIENTS_COL = "assigned_fil_max_ideal_clients"

MAX_VALID_TIME_MIN = 7000.0

# =============================================================================
# ONSTOVE DEMAND DEFAULTS (kept explicit for visibility)
# =============================================================================
MEALS_PER_DAY = 1.0
ENERGY_PER_MEAL_MJ = 3.64
EFFICIENCY = 0.5
ENERGY_CONTENT_MJ_PER_KG = 45.5

# =============================================================================
# RESELLER TRANSPORT MODEL PARAMETERS (from paper / assumptions)
# =============================================================================
truck_capacity_kg = 9996
utilization_factor = 0.85
truck_speed_kmh = 30
variable_cost_per_km = 0.540
driver_annual_salary_usd = 2604
salary_multiplier = 1.18
hours_per_day = 8
discount_rate = 0.10
truck_overnight_cost_usd = 170200  # add source
truck_life_years = 15
license_cost_5y_usd = 418
days_per_year = 330
fixed_loading_unloading_hours = 1.0

# =============================================================================
# FILLING STATION COST PARAMETERS (Gurkan et al. 2026)
# =============================================================================
DISCOUNT_RATE = 0.10
FOM_ANNUAL_USD = 384_400  # Gurkan et al. (2026)
VOM_USD_PER_KG = 0.035  # Gurkan et al. (2026)
OP_LICENSE_FEE_USD = 3952
OP_LICENSE_VALIDITY_YEARS = 3
REFERENCE_FILLING_CAPACITY_KG = 35_000_000  # 35,000 ton/year (Gurkan et al. 2026)
COST_INCREASE_CAP_PCT = 0.30  # cap out cost to +30% of corresponding in cost

def annual_lpg_demand_kg(
    num_clients: pd.Series | np.ndarray | float,
    meals_per_day: float = MEALS_PER_DAY,
    energy_per_meal: float = ENERGY_PER_MEAL_MJ,
    efficiency: float = EFFICIENCY,
    energy_content: float = ENERGY_CONTENT_MJ_PER_KG,
) -> pd.Series | np.ndarray | float:
    annual_energy_mj = num_clients * meals_per_day * 365.0 * energy_per_meal
    annual_mass_kg = annual_energy_mj / (efficiency * energy_content)
    return annual_mass_kg

def crf(rate: float, years: int) -> float:
    return (rate * (1.0 + rate) ** years) / ((1.0 + rate) ** years - 1.0)

effective_load_kg = truck_capacity_kg * utilization_factor
driver_hourly_cost_usd = (
    driver_annual_salary_usd * salary_multiplier / (hours_per_day * days_per_year)
)
crf_truck = crf(discount_rate, truck_life_years)
crf_license = crf(discount_rate, 5)
annual_capital_cost_usd = truck_overnight_cost_usd * crf_truck
annual_license_cost_usd = license_cost_5y_usd * crf_license
fixed_annual_truck_cost_usd = annual_capital_cost_usd + annual_license_cost_usd

crf_op = crf(DISCOUNT_RATE, OP_LICENSE_VALIDITY_YEARS)
annual_op_license_cost_usd = OP_LICENSE_FEE_USD * crf_op
TOTAL_FIXED_ANNUAL_USD = FOM_ANNUAL_USD + annual_op_license_cost_usd
FIXED_COST_PER_KG_FILL = TOTAL_FIXED_ANNUAL_USD / REFERENCE_FILLING_CAPACITY_KG

print("[1/6] Loading required layers...")
resell = gpd.read_file(CHAIN_GPKG, layer=RESELL_LAYER)
filling = gpd.read_file(CHAIN_GPKG, layer=FILLING_LAYER)
sell_points = gpd.read_file(SELL_POINT_GPKG, layer=SELL_POINT_LAYER)
filling_points = gpd.read_file(FILLING_POINT_GPKG, layer=FILLING_POINT_LAYER)

if resell.empty:
    raise RuntimeError(f"Layer '{RESELL_LAYER}' is empty.")
if filling.empty:
    raise RuntimeError(f"Layer '{FILLING_LAYER}' is empty.")
if sell_points.empty:
    raise RuntimeError(f"Layer '{SELL_POINT_LAYER}' is empty.")
if filling_points.empty:
    raise RuntimeError(f"Layer '{FILLING_POINT_LAYER}' is empty.")

for c in [ID_COL, FILL_REF_COL, FILL_TIME_COL]:
    if c not in resell.columns:
        raise KeyError(f"Missing column '{c}' in layer '{RESELL_LAYER}'.")
for c in [ID_COL, FILL_COST_IN_COL]:
    if c not in filling.columns:
        raise KeyError(f"Missing column '{c}' in layer '{FILLING_LAYER}'.")
for c in [ID_COL, SELL_CLIENTS_COL]:
    if c not in sell_points.columns:
        raise KeyError(f"Missing column '{c}' in layer '{SELL_POINT_LAYER}'.")
# Check required columns in filling_points (including 'clients' for direct clients)
for c in [ID_COL, FILL_TOTAL_CLIENTS_COL, "clients"]:
    if c not in filling_points.columns:
        raise KeyError(f"Missing column '{c}' in layer '{FILLING_POINT_LAYER}'.")

print("[2/6] Building demand maps and computing filling outbound cost...")

# Build robust filling->clients map directly from filling_point_clients
fp = filling_points[[ID_COL, FILL_TOTAL_CLIENTS_COL, "clients"]].copy()
fp[ID_COL] = pd.to_numeric(fp[ID_COL], errors="coerce")
fp[FILL_TOTAL_CLIENTS_COL] = pd.to_numeric(fp[FILL_TOTAL_CLIENTS_COL], errors="coerce")
fp["clients"] = pd.to_numeric(fp["clients"], errors="coerce")
fp = fp[fp[ID_COL].notna() & (fp[ID_COL] > 0)].copy()

# Compute assigned (transported) clients = total - direct
fp["assigned_clients"] = fp[FILL_TOTAL_CLIENTS_COL] - fp["clients"]
fp["assigned_clients"] = fp["assigned_clients"].clip(lower=0)  # Ensure non-negative

n_fp_raw = len(fp)
n_fp_unique = int(fp[ID_COL].nunique())
if n_fp_unique < n_fp_raw:
    fp_agg = (
        fp.groupby(ID_COL, as_index=False)[[FILL_TOTAL_CLIENTS_COL, "assigned_clients"]]
        .sum(min_count=1)
        .fillna({FILL_TOTAL_CLIENTS_COL: 0.0, "assigned_clients": 0.0})
    )
    fp_agg[ID_COL] = fp_agg[ID_COL].astype(np.int64)
    print(f"- Detected duplicate filling IDs in {FILLING_POINT_LAYER}; aggregated {n_fp_raw - n_fp_unique:,} duplicate rows.")
else:
    fp_agg = fp.copy()
    fp_agg[ID_COL] = fp_agg[ID_COL].astype(np.int64)
    fp_agg[FILL_TOTAL_CLIENTS_COL] = fp_agg[FILL_TOTAL_CLIENTS_COL].fillna(0.0).astype(float)
    fp_agg["assigned_clients"] = fp_agg["assigned_clients"].fillna(0.0).astype(float)

fillp_id = pd.to_numeric(fp_agg[ID_COL], errors="coerce")
fillp_total_clients = pd.to_numeric(fp_agg[FILL_TOTAL_CLIENTS_COL], errors="coerce").fillna(0.0).astype(float)
fillp_assigned_clients = pd.to_numeric(fp_agg["assigned_clients"], errors="coerce").fillna(0.0).astype(float)
mask_fillp = fillp_id.notna()
map_filling_total_clients = dict(zip(fillp_id[mask_fillp].astype(np.int64), fillp_total_clients[mask_fillp].astype(float)))
map_filling_assigned_clients = dict(zip(fillp_id[mask_fillp].astype(np.int64), fillp_assigned_clients[mask_fillp].astype(float)))

filling[ID_COL] = pd.to_numeric(filling[ID_COL], errors="coerce")
filling_cost_in = pd.to_numeric(filling[FILL_COST_IN_COL], errors="coerce").astype(float)

# --- Uniform fixed cost per kg for filling station ---
# Source: Gurkan et al. (2026) – FOM 384,400 USD/y, license 3,952 USD/3y, VOM 0.035 USD/kg.
fixed_cost_per_kg_fill = np.full(len(filling), FIXED_COST_PER_KG_FILL, dtype=float)
added_cost_per_kg_fill = fixed_cost_per_kg_fill + VOM_USD_PER_KG

valid_fill = (
    np.isfinite(filling_cost_in) & (filling_cost_in >= 0)
    # annual_kg_fill is not used for fixed cost anymore
    )

new_filling_out_uncapped = np.where(valid_fill, filling_cost_in.to_numpy(dtype=float) + added_cost_per_kg_fill, np.nan)
max_filling_out_allowed = filling_cost_in.to_numpy(dtype=float) * (1.0 + COST_INCREASE_CAP_PCT)
fill_capped_mask = valid_fill & np.isfinite(new_filling_out_uncapped) & (new_filling_out_uncapped > max_filling_out_allowed)
new_filling_out = np.where(valid_fill, np.minimum(new_filling_out_uncapped, max_filling_out_allowed), np.nan)

# NOTE: Direct clients of a filling point are assumed to pay the wholesale price
# (cost_fil_kg_out). In reality, the filling point incurs retailing costs similar to
# a reseller, so a markup could be added. The current approach effectively treats
# filling points as lower-cost outlets.
filling[FILL_COST_OUT_COL] = pd.to_numeric(pd.Series(new_filling_out, index=filling.index), errors="coerce").astype(float)

filling = gpd.GeoDataFrame(filling, geometry="geometry", crs=filling.crs)
filling.to_file(CHAIN_GPKG, layer=FILLING_LAYER, driver="GPKG")

n_fill = int(len(filling))
n_fill_valid = int(valid_fill.sum())
over_capacity = valid_fill & (False)  # annual_kg_fill not used
print(f"Filling updated in {CHAIN_GPKG} | layer={FILLING_LAYER}")
print(f"- Valid filling rows: {n_fill_valid:,}/{n_fill:,}")
print(f"- Filling points capped at +{100.0 * COST_INCREASE_CAP_PCT:.0f}%: {int(fill_capped_mask.sum()):,}")
if n_fill_valid > 0:
    c = filling.loc[valid_fill, FILL_COST_OUT_COL].to_numpy(dtype=float)
    print(
        f"- {FILL_COST_OUT_COL} min/median/mean/p95/max: ",
        f"{float(np.nanmin(c)):.4f} / {float(np.nanmedian(c)):.4f} / ",
        f"{float(np.nanmean(c)):.4f} / {float(np.nanpercentile(c, 95)):.4f} / {float(np.nanmax(c)):.4f}")
if np.any(over_capacity):
    print(f"WARNING: {int(over_capacity.sum()):,} filling point(s) exceed {REFERENCE_FILLING_CAPACITY_KG:,.0f} kg/year.")

print("[3/6] Building reference maps (costs and demand)...")
fid = pd.to_numeric(filling[ID_COL], errors="coerce")
fcost_out = pd.to_numeric(filling[FILL_COST_OUT_COL], errors="coerce")
mask_fill = fid.notna()
map_filling_cost_out = dict(zip(fid[mask_fill].astype(np.int64), fcost_out[mask_fill].astype(float)))

sell_id = pd.to_numeric(sell_points[ID_COL], errors="coerce")
sell_clients = pd.to_numeric(sell_points[SELL_CLIENTS_COL], errors="coerce")
sell_clients = sell_clients.fillna(0.0).astype(float)
mask_sell = sell_id.notna()
map_reseller_clients = dict(zip(sell_id[mask_sell].astype(np.int64), sell_clients[mask_sell].astype(float)))

print("[4/6] Computing reseller inbound cost...")
rid = pd.to_numeric(resell[ID_COL], errors="coerce")
fref = pd.to_numeric(resell[FILL_REF_COL], errors="coerce")
tmin = pd.to_numeric(resell[FILL_TIME_COL], errors="coerce")

fill_cost_out_ref = fref.map(map_filling_cost_out)
reseller_clients = rid.map(map_reseller_clients)

# --- TRUCK TRANSPORT COST (uniform allocation, 2026 update) ---
# Source: Gurkan et al. (2026)
# Use actual one-way distance from resell['filling_ref_distance_km']
# Variable cost per kg: uses actual round-trip distance and time
# Fixed cost per kg: based on reference annual truck utilization and average round-trip distance
TRUCK_ANNUAL_KM = 5000.0  # km/year (Gurkan et al. 2026)
if 'filling_ref_distance_km' not in resell.columns:
    raise KeyError("Missing 'filling_ref_distance_km' in reseller layer. Run 4.4 to add this column.")
dist_km = pd.to_numeric(resell['filling_ref_distance_km'], errors='coerce').to_numpy(dtype=float)
if not np.any(np.isfinite(dist_km)):
    raise RuntimeError("All filling_ref_distance_km values are NaN or invalid. Check 4.4 output.")
avg_one_way_dist_km = np.nanmean(dist_km[np.isfinite(dist_km)])
if avg_one_way_dist_km <= 0:
    raise RuntimeError("Average one-way distance for truck is zero or negative. Check data.")
avg_round_trip_km = 2.0 * avg_one_way_dist_km
trips_per_year = TRUCK_ANNUAL_KM / avg_round_trip_km
if trips_per_year <= 0:
    raise RuntimeError("Computed trips_per_year is zero or negative. Check avg_round_trip_km and TRUCK_ANNUAL_KM.")
fixed_truck_cost_per_kg = fixed_annual_truck_cost_usd / (effective_load_kg * trips_per_year)

# Variable cost per kg (per reseller)
time_min = tmin.to_numpy(dtype=float)
round_trip_hours = (time_min * 2.0 / 60.0) + fixed_loading_unloading_hours
round_trip_distance_km = 2.0 * dist_km
variable_cost_trip = (variable_cost_per_km * round_trip_distance_km + driver_hourly_cost_usd * round_trip_hours)
variable_cost_per_kg = variable_cost_trip / effective_load_kg

transport_cost_kg = fixed_truck_cost_per_kg + variable_cost_per_kg

annual_demand_reseller_kg = pd.to_numeric(
    annual_lpg_demand_kg(reseller_clients), errors="coerce"
    ).astype(float)

valid = (
    fref.notna()
    & (fref > 0)
    & np.isfinite(time_min)
    & (time_min >= 0)
    & (time_min < MAX_VALID_TIME_MIN)
    & fill_cost_out_ref.notna()
    & np.isfinite(fill_cost_out_ref)
    & (fill_cost_out_ref >= 0)
    & annual_demand_reseller_kg.notna()
    & np.isfinite(annual_demand_reseller_kg)
    & (annual_demand_reseller_kg > 0)
    & np.isfinite(transport_cost_kg)
    & (transport_cost_kg >= 0)
    & np.isfinite(dist_km)
    & (dist_km > 0)
    )

new_cost_in = np.where(
    valid,
    fill_cost_out_ref + transport_cost_kg,
    np.nan,
    )

print("[5/6] Overwriting reseller inbound cost in chain_with_cost.gpkg...")
resell[FILL_ORIGIN_COST_OUT_COL] = pd.to_numeric(fill_cost_out_ref, errors="coerce").astype(float)
resell[RESELL_COST_IN_COL] = np.nan
resell.loc[valid, RESELL_COST_IN_COL] = pd.to_numeric(
    pd.Series(new_cost_in, index=resell.index)[valid], errors="coerce"
    ).astype(float)
resell[RESELL_COST_IN_COL] = pd.to_numeric(resell[RESELL_COST_IN_COL], errors="coerce").astype(float)
resell = gpd.GeoDataFrame(resell, geometry="geometry", crs=resell.crs)
resell.to_file(CHAIN_GPKG, layer=RESELL_LAYER, driver="GPKG")

print("[6/6] QA summary and NaN diagnostics...")
n_total = int(len(resell))
n_valid = int(valid.sum())
n_nan = int((~np.isfinite(resell[RESELL_COST_IN_COL])).sum())

nan_due_to_ref = int((~(fref.notna() & (fref > 0) & fill_cost_out_ref.notna() & np.isfinite(fill_cost_out_ref))).sum())
nan_due_to_time = int((~(np.isfinite(time_min) & (time_min >= 0) & (time_min < MAX_VALID_TIME_MIN))).sum())
nan_due_to_reseller_demand = int((~(annual_demand_reseller_kg.notna() & np.isfinite(annual_demand_reseller_kg) & (annual_demand_reseller_kg > 0))).sum())
zero_or_missing_transported = 0  # not used in new logic

print(f"Updated rows (valid): {n_valid:,}/{n_total:,}")
print(f"NaN rows in {RESELL_COST_IN_COL}: {n_nan:,}")
print("NaN diagnostics (overlapping causes, not mutually exclusive):")
print(f"- invalid/missing filling reference or filling cost source: {nan_due_to_ref:,}")
print(f"- invalid travel time (<0 or >= {MAX_VALID_TIME_MIN:.0f} or NaN): {nan_due_to_time:,}")
print(f"- invalid/missing reseller demand (from sell_point_clients.clients): {nan_due_to_reseller_demand:,}")

if n_valid > 0:
    s = pd.to_numeric(resell.loc[valid, RESELL_COST_IN_COL], errors="coerce").astype(float)
    print(
        "Sanity (valid rows) min/median/mean/p95/max: ",
        f"{float(np.nanmin(s)):.4f} / ",
        f"{float(np.nanmedian(s)):.4f} / ",
        f"{float(np.nanmean(s)):.4f} / ",
        f"{float(np.nanpercentile(s, 95)):.4f} / ",
        f"{float(np.nanmax(s)):.4f}")
else:
    raise RuntimeError("No valid reseller rows found for advanced inbound cost update.")

print(f"Output file: {CHAIN_GPKG}")
print(f"Column overwritten: {RESELL_COST_IN_COL}")
print(f"Kept helper column: {FILL_ORIGIN_COST_OUT_COL}")

[1/6] Loading required layers...
[2/6] Building demand maps and computing filling outbound cost...
Filling updated in dataset_big\chain_with_cost.gpkg | layer=filling
- Valid filling rows: 375/375
- Filling points capped at +30%: 0
- cost_fil_kg_out min/median/mean/p95/max:  0.5460 / 0.5460 /  0.5460 / 0.5460 / 0.5460
[3/6] Building reference maps (costs and demand)...
[4/6] Computing reseller inbound cost...
[5/6] Overwriting reseller inbound cost in chain_with_cost.gpkg...
[6/6] QA summary and NaN diagnostics...
Updated rows (valid): 1,832/2,413
NaN rows in cost_res_kg_in: 581
NaN diagnostics (overlapping causes, not mutually exclusive):
- invalid/missing filling reference or filling cost source: 0
- invalid travel time (<0 or >= 7000 or NaN): 0
- invalid/missing reseller demand (from sell_point_clients.clients): 504
Sanity (valid rows) min/median/mean/p95/max:  0.5746 /  0.5753 /  0.5784 /  0.5913 /  0.6342
Output file: dataset_big\chain_with_cost.gpkg
Column overwritten: cost_res_k

In [2]:
"""
Compute reseller outbound cost (cost_res_kg_out) from inbound cost using uniform fixed cost allocation (2026 update).
- Fixed cost per kg is based on system-average reseller demand (excluding direct filling clients).
- Retailer fixed costs: rent 50 USD/m, salary 60 USD/m, license 20 USD/y (placeholders – validate locally).
- Source: see docstring and inline comments.
"""

from __future__ import annotations

from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd

# =============================================================================
# PATHS AND LAYERS
# =============================================================================
DATA_DIR = Path("dataset_big")
CHAIN_GPKG = DATA_DIR / "chain_with_cost.gpkg"
SELL_CLIENTS_GPKG = DATA_DIR / "sell_point_clients.gpkg"

RESELL_LAYER = "resell"
SELL_CLIENTS_LAYER = "sell_point_clients"

ID_COL = "id_res&fil"
RESELL_COST_IN_COL = "cost_res_kg_in"
RESELL_COST_OUT_COL = "cost_res_kg_out"
CLIENTS_COL = "clients"

# =============================================================================
# DEMAND MODEL PARAMETERS (same structure used in this pipeline)
# =============================================================================
MEALS_PER_DAY = 1.0
ENERGY_PER_MEAL_MJ = 3.64
EFFICIENCY = 0.5
ENERGY_CONTENT_MJ_PER_KG = 45.5

def annual_lpg_demand_kg(num_clients: np.ndarray | float) -> np.ndarray | float:
    annual_energy_mj = num_clients * MEALS_PER_DAY * 365.0 * ENERGY_PER_MEAL_MJ
    annual_mass_kg = annual_energy_mj / (EFFICIENCY * ENERGY_CONTENT_MJ_PER_KG)
    return annual_mass_kg

# =============================================================================
# RETAILER FIXED COST ASSUMPTIONS (PLACEHOLDERS - VALIDATE LOCALLY)
# =============================================================================
# Monthly costs in USD
RENT_PER_MONTH_USD = 50.0
SALARY_PER_MONTH_USD = 60.0
LICENSE_ANNUAL_USD = 20.0

# Annualize fixed costs
ANNUAL_RENT_USD = RENT_PER_MONTH_USD * 12.0
ANNUAL_SALARY_USD = SALARY_PER_MONTH_USD * 12.0
TOTAL_FIXED_ANNUAL_USD = ANNUAL_RENT_USD + ANNUAL_SALARY_USD + LICENSE_ANNUAL_USD

# Optional variable O&M for reseller point (kept explicit; set to zero for now)
VOM_USD_PER_KG = 0.0
COST_INCREASE_CAP_PCT = 0.30  # cap out cost to +30% of inbound reseller cost

print("[1/4] Loading reseller layer and clients data...")
resell = gpd.read_file(CHAIN_GPKG, layer=RESELL_LAYER)
clients_gdf = gpd.read_file(SELL_CLIENTS_GPKG, layer=SELL_CLIENTS_LAYER)

if resell.empty:
    raise RuntimeError(f"Layer '{RESELL_LAYER}' is empty.")
if clients_gdf.empty:
    raise RuntimeError(f"Layer '{SELL_CLIENTS_LAYER}' is empty.")

for col, layer in [(ID_COL, RESELL_LAYER), (RESELL_COST_IN_COL, RESELL_LAYER)]:
    if col not in resell.columns:
        raise KeyError(f"Missing '{col}' in layer '{layer}'")
for col in [ID_COL, CLIENTS_COL]:
    if col not in clients_gdf.columns:
        raise KeyError(f"Missing '{col}' in layer '{SELL_CLIENTS_LAYER}'")

# Aggregate clients by reseller ID to make merge robust to duplicates in source
clients_tab = clients_gdf[[ID_COL, CLIENTS_COL]].copy()
clients_tab[ID_COL] = pd.to_numeric(clients_tab[ID_COL], errors="coerce")
clients_tab[CLIENTS_COL] = pd.to_numeric(clients_tab[CLIENTS_COL], errors="coerce")
clients_tab = clients_tab[clients_tab[ID_COL].notna() & (clients_tab[ID_COL] > 0)].copy()
clients_tab = (
    clients_tab.groupby(ID_COL, as_index=False)[CLIENTS_COL]
    .sum(min_count=1)
    .fillna({CLIENTS_COL: 0.0})
)

resell[ID_COL] = pd.to_numeric(resell[ID_COL], errors="coerce")
resell = resell.merge(clients_tab, on=ID_COL, how="left")
resell[CLIENTS_COL] = pd.to_numeric(resell[CLIENTS_COL], errors="coerce").fillna(0.0)

print("[2/4] Computing annual demand and added cost per kg...")
total_clients = resell[CLIENTS_COL].to_numpy(dtype=float)
annual_kg = annual_lpg_demand_kg(total_clients)
cost_in = pd.to_numeric(resell[RESELL_COST_IN_COL], errors="coerce").to_numpy(dtype=float)

# --- Uniform fixed cost per kg for reseller ---
# Compute total reseller demand (excluding direct filling clients)
# (Assume sell_points includes only resellers, not direct filling clients)
total_reseller_demand_kg = np.nansum(annual_kg)
n_resellers = np.sum(~np.isnan(annual_kg) & (annual_kg > 0))
if n_resellers == 0 or total_reseller_demand_kg <= 0:
    raise RuntimeError("No valid resellers or total demand is zero for fixed cost allocation.")
avg_reseller_demand_kg = total_reseller_demand_kg / n_resellers
FIXED_RETAILER_COST_PER_KG = TOTAL_FIXED_ANNUAL_USD / avg_reseller_demand_kg

fixed_cost_per_kg = np.full_like(annual_kg, FIXED_RETAILER_COST_PER_KG, dtype=float)
added_cost_per_kg = fixed_cost_per_kg + VOM_USD_PER_KG

valid_mask = np.isfinite(annual_kg) & (annual_kg > 0) & np.isfinite(cost_in) & (cost_in >= 0)

cost_out_uncapped = np.where(valid_mask, cost_in + added_cost_per_kg, np.nan)
max_cost_out_allowed = cost_in * (1.0 + COST_INCREASE_CAP_PCT)
res_capped_mask = valid_mask & np.isfinite(cost_out_uncapped) & (cost_out_uncapped > max_cost_out_allowed)
cost_out = np.where(valid_mask, np.minimum(cost_out_uncapped, max_cost_out_allowed), np.nan)

print("[3/4] Updating reseller layer...")
resell[RESELL_COST_OUT_COL] = pd.to_numeric(pd.Series(cost_out, index=resell.index), errors="coerce").astype(float)
# Helper column for diagnostics and sensitivity checks
resell["res_added_fixed_usd_kg"] = pd.to_numeric(pd.Series(fixed_cost_per_kg, index=resell.index), errors="coerce").astype(float)

resell = gpd.GeoDataFrame(resell, geometry="geometry", crs=resell.crs)
resell.to_file(CHAIN_GPKG, layer=RESELL_LAYER, driver="GPKG")

print("[4/4] QA summary...")
n_total = len(resell)
n_valid = int(valid_mask.sum())
n_no_demand = int(np.sum(np.isfinite(annual_kg) & (annual_kg <= 0)))
n_invalid_inbound = int(np.sum(~(np.isfinite(cost_in) & (cost_in >= 0))))
print(f"Reseller points total                    : {n_total:,}")
print(f"Reseller points with valid in+out inputs : {n_valid:,}")
print(f"Reseller points capped at +{100.0 * COST_INCREASE_CAP_PCT:.0f}%: {int(res_capped_mask.sum()):,}")
print(f"Rows with zero/nonpositive annual demand : {n_no_demand:,}")
print(f"Rows with invalid inbound cost           : {n_invalid_inbound:,}")

if n_valid > 0:
    out_valid = resell.loc[valid_mask, RESELL_COST_OUT_COL].to_numpy(dtype=float)
    add_valid = added_cost_per_kg[valid_mask]
    print(
        f"{RESELL_COST_OUT_COL} min/median/mean/max: ",
        f"{np.nanmin(out_valid):.4f} / {np.nanmedian(out_valid):.4f} / ",
        f"{np.nanmean(out_valid):.4f} / {np.nanmax(out_valid):.4f}")
    print(
        "Added retailer cost min/median/mean/max: ",
        f"{np.nanmin(add_valid):.4f} / {np.nanmedian(add_valid):.4f} / ",
        f"{np.nanmean(add_valid):.4f} / {np.nanmax(add_valid):.4f}")
else:
    print("No valid reseller rows with annual demand > 0. cost_res_kg_out left as NaN.")

print(f"Done. Updated layer '{RESELL_LAYER}' in {CHAIN_GPKG}")

[1/4] Loading reseller layer and clients data...
[2/4] Computing annual demand and added cost per kg...
[3/4] Updating reseller layer...
[4/4] QA summary...
Reseller points total                    : 2,413
Reseller points with valid in+out inputs : 1,832
Reseller points capped at +30%: 0
Rows with zero/nonpositive annual demand : 504
Rows with invalid inbound cost           : 581
cost_res_kg_out min/median/mean/max:  0.5766 / 0.5772 /  0.5804 / 0.6362
Added retailer cost min/median/mean/max:  0.0020 / 0.0020 /  0.0020 / 0.0020
Done. Updated layer 'resell' in dataset_big\chain_with_cost.gpkg


In [3]:
"""
Final QA stats for 4.5 outputs with split in/out costs.

Checks:
1) filling points have valid cost_fil_kg_out and cost_fil_kg_in
2) resell points have valid filling_reference and travel stats
3) resell points have valid cost_res_kg_in and cost_res_kg_out
4) reseller outbound premium is coherent with fixed-cost add-on logic
"""

from __future__ import annotations

from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd

DATA_DIR = Path("dataset_big")
WORK_GPKG = DATA_DIR / "chain_with_cost.gpkg"

RESELL_LAYER = "resell"
FILLING_LAYER = "filling"
ID_COL = "id_res&fil"
FILL_REF_COL = "filling_reference"
FILL_TIME_COL = "filling_ref_time_min"
FILL_COST_OUT_COL = "cost_fil_kg_out"
FILL_COST_IN_COL = "cost_fil_kg_in"
RESELL_COST_IN_COL = "cost_res_kg_in"
RESELL_COST_OUT_COL = "cost_res_kg_out"
RESELL_ADDED_FIXED_COL = "res_added_fixed_usd_kg"
LEGACY_COST_COL = "cost_kg"

print("[1/4] Loading layers...")
resell = gpd.read_file(WORK_GPKG, layer=RESELL_LAYER)
filling = gpd.read_file(WORK_GPKG, layer=FILLING_LAYER)
if resell.empty or filling.empty:
    raise RuntimeError("Resell or filling layer is empty in chain_with_cost.gpkg")

for c in [ID_COL, FILL_COST_OUT_COL, FILL_COST_IN_COL]:
    if c not in filling.columns:
        raise KeyError(f"Missing column '{c}' in layer '{FILLING_LAYER}'")
for c in [ID_COL, FILL_REF_COL, FILL_TIME_COL, RESELL_COST_IN_COL, RESELL_COST_OUT_COL]:
    if c not in resell.columns:
        raise KeyError(f"Missing column '{c}' in layer '{RESELL_LAYER}'")

fill_id = pd.to_numeric(filling[ID_COL], errors="coerce")
fill_cost_out = pd.to_numeric(filling[FILL_COST_OUT_COL], errors="coerce")
fill_cost_in = pd.to_numeric(filling[FILL_COST_IN_COL], errors="coerce")
res_ref = pd.to_numeric(resell[FILL_REF_COL], errors="coerce")
res_tmin = pd.to_numeric(resell[FILL_TIME_COL], errors="coerce")
res_cost_in = pd.to_numeric(resell[RESELL_COST_IN_COL], errors="coerce")
res_cost_out = pd.to_numeric(resell[RESELL_COST_OUT_COL], errors="coerce")

print("[2/4] QA on filling in/out costs...")
n_fill = len(filling)
fill_out_valid = np.isfinite(fill_cost_out)
fill_in_valid = np.isfinite(fill_cost_in)
fill_out_nonneg = fill_out_valid & (fill_cost_out >= 0)
fill_in_nonneg = fill_in_valid & (fill_cost_in >= 0)

print("=== FILLING COST QA ===")
print(f"Filling points total                  : {n_fill:,}")
print(f"Finite {FILL_COST_OUT_COL}            : {int(fill_out_valid.sum()):,}")
print(f"Finite {FILL_COST_IN_COL}             : {int(fill_in_valid.sum()):,}")
print(f"{FILL_COST_OUT_COL} < 0               : {int((~fill_out_nonneg).sum()):,}")
print(f"{FILL_COST_IN_COL} < 0                : {int((~fill_in_nonneg).sum()):,}")
if fill_out_valid.any():
    print(
        f"{FILL_COST_OUT_COL} min/median/mean/max: "
        f"{float(np.nanmin(fill_cost_out)):.4f} / {float(np.nanmedian(fill_cost_out)):.4f} / "
        f"{float(np.nanmean(fill_cost_out)):.4f} / {float(np.nanmax(fill_cost_out)):.4f}"
    )
if fill_in_valid.any():
    print(
        f"{FILL_COST_IN_COL} min/median/mean/max : "
        f"{float(np.nanmin(fill_cost_in)):.4f} / {float(np.nanmedian(fill_cost_in)):.4f} / "
        f"{float(np.nanmean(fill_cost_in)):.4f} / {float(np.nanmax(fill_cost_in)):.4f}"
    )
if LEGACY_COST_COL in filling.columns:
    legacy_fill = pd.to_numeric(filling[LEGACY_COST_COL], errors="coerce")
    print(f"Legacy {LEGACY_COST_COL} finite count : {int(np.isfinite(legacy_fill).sum()):,}/{n_fill:,}")

print("\n[3/4] QA on reseller filling reference, travel stats, and costs...")
n_res = len(resell)
fill_id_set = set(fill_id[np.isfinite(fill_id)].astype(np.int64).tolist())
ref_finite = np.isfinite(res_ref)
ref_positive = ref_finite & (res_ref > 0)
ref_exists = ref_positive & pd.Series(res_ref.astype("Int64")).isin(fill_id_set).to_numpy()

time_valid = np.isfinite(res_tmin) & (res_tmin >= 0) & (res_tmin < 7000)
ref_and_time_valid = ref_exists & time_valid

res_in_valid = np.isfinite(res_cost_in) & (res_cost_in >= 0)
res_out_valid = np.isfinite(res_cost_out) & (res_cost_out >= 0)
res_in_linked = ref_and_time_valid & res_in_valid
res_out_linked = ref_and_time_valid & res_out_valid

print("=== RESELLER REFERENCE QA ===")
print(f"Resell points total                    : {n_res:,}")
print(f"Resell with finite filling_reference   : {int(ref_finite.sum()):,}")
print(f"Resell with existing filling_reference : {int(ref_exists.sum()):,}")
print(f"Resell with valid reference + time     : {int(ref_and_time_valid.sum()):,}")
print(f"Missing/invalid reference              : {int((~ref_exists).sum()):,}")
if ref_and_time_valid.any():
    t = res_tmin[ref_and_time_valid].to_numpy(dtype=float)
    print(
        "Travel time min/median/mean/p95/max (min): "
        f"{float(np.nanmin(t)):.2f} / {float(np.nanmedian(t)):.2f} / {float(np.nanmean(t)):.2f} / "
        f"{float(np.nanpercentile(t, 95)):.2f} / {float(np.nanmax(t)):.2f}"
    )

print("=== RESELLER COST QA ===")
print(f"Finite {RESELL_COST_IN_COL}                    : {int(np.isfinite(res_cost_in).sum()):,}/{n_res:,}")
print(f"Finite {RESELL_COST_OUT_COL}                   : {int(np.isfinite(res_cost_out).sum()):,}/{n_res:,}")
print(f"{RESELL_COST_IN_COL} valid + linked            : {int(res_in_linked.sum()):,}/{n_res:,}")
print(f"{RESELL_COST_OUT_COL} valid + linked           : {int(res_out_linked.sum()):,}/{n_res:,}")
print(f"{RESELL_COST_IN_COL} NaN rows                  : {int((~np.isfinite(res_cost_in)).sum()):,}")
print(f"{RESELL_COST_OUT_COL} NaN rows                 : {int((~np.isfinite(res_cost_out)).sum()):,}")
if res_in_valid.any():
    print(
        f"{RESELL_COST_IN_COL} min/median/mean/p95/max      : "
        f"{float(np.nanmin(res_cost_in)):.4f} / {float(np.nanmedian(res_cost_in)):.4f} / "
        f"{float(np.nanmean(res_cost_in)):.4f} / {float(np.nanpercentile(res_cost_in, 95)):.4f} / "
        f"{float(np.nanmax(res_cost_in)):.4f}"
    )
if res_out_valid.any():
    print(
        f"{RESELL_COST_OUT_COL} min/median/mean/p95/max     : "
        f"{float(np.nanmin(res_cost_out)):.4f} / {float(np.nanmedian(res_cost_out)):.4f} / "
        f"{float(np.nanmean(res_cost_out)):.4f} / {float(np.nanpercentile(res_cost_out, 95)):.4f} / "
        f"{float(np.nanmax(res_cost_out)):.4f}"
    )

paired_valid = res_in_valid & res_out_valid & ref_and_time_valid
if paired_valid.any():
    premium = (res_cost_out[paired_valid] - res_cost_in[paired_valid]).to_numpy(dtype=float)
    monotonic_ok = int(np.sum(premium >= -1e-9))
    print("=== OUTBOUND PREMIUM QA ===")
    print(f"Rows with out >= in (paired valid)     : {monotonic_ok:,}/{int(paired_valid.sum()):,}")
    print(
        "Premium (out - in) min/median/mean/p95/max: "
        f"{float(np.nanmin(premium)):.6f} / {float(np.nanmedian(premium)):.6f} / "
        f"{float(np.nanmean(premium)):.6f} / {float(np.nanpercentile(premium, 95)):.6f} / "
        f"{float(np.nanmax(premium)):.6f}"
    )

if RESELL_ADDED_FIXED_COL in resell.columns:
    added = pd.to_numeric(resell[RESELL_ADDED_FIXED_COL], errors="coerce")
    added_valid = np.isfinite(added) & (added >= 0)
    print("=== FIXED ADD-ON QA ===")
    print(f"Finite {RESELL_ADDED_FIXED_COL}            : {int(added_valid.sum()):,}/{n_res:,}")
    if added_valid.any():
        av = added[added_valid].to_numpy(dtype=float)
        print(
            f"{RESELL_ADDED_FIXED_COL} min/median/mean/p95/max: "
            f"{float(np.nanmin(av)):.6f} / {float(np.nanmedian(av)):.6f} / "
            f"{float(np.nanmean(av)):.6f} / {float(np.nanpercentile(av, 95)):.6f} / {float(np.nanmax(av)):.6f}"
        )

if (resell.crs is not None) and (filling.crs is not None) and (str(resell.crs) == str(filling.crs)):
    fill_geom = filling[[ID_COL, "geometry"]].copy()
    fill_geom[ID_COL] = pd.to_numeric(fill_geom[ID_COL], errors="coerce")
    merged = resell[[ID_COL, FILL_REF_COL, "geometry"]].copy()
    merged[ID_COL] = pd.to_numeric(merged[ID_COL], errors="coerce")
    merged[FILL_REF_COL] = pd.to_numeric(merged[FILL_REF_COL], errors="coerce")
    merged = merged.merge(fill_geom, left_on=FILL_REF_COL, right_on=ID_COL, how="left", suffixes=("_res", "_fill"))
    good_geom = merged["geometry_res"].notna() & merged["geometry_fill"].notna()
    if good_geom.any():
        dist_km = merged.loc[good_geom, "geometry_res"].distance(merged.loc[good_geom, "geometry_fill"]).astype(float) / 1000.0
        print(
            "Straight-line distance min/median/mean/p95/max (km): "
            f"{float(np.nanmin(dist_km)):.2f} / {float(np.nanmedian(dist_km)):.2f} / {float(np.nanmean(dist_km)):.2f} / "
            f"{float(np.nanpercentile(dist_km, 95)):.2f} / {float(np.nanmax(dist_km)):.2f}"
        )

print("\n[4/4] QA completed.")

[1/4] Loading layers...
[2/4] QA on filling in/out costs...
=== FILLING COST QA ===
Filling points total                  : 375
Finite cost_fil_kg_out            : 375
Finite cost_fil_kg_in             : 375
cost_fil_kg_out < 0               : 0
cost_fil_kg_in < 0                : 0
cost_fil_kg_out min/median/mean/max: 0.5460 / 0.5460 / 0.5460 / 0.5460
cost_fil_kg_in min/median/mean/max : 0.5000 / 0.5000 / 0.5000 / 0.5000
Legacy cost_kg finite count : 375/375

[3/4] QA on reseller filling reference, travel stats, and costs...
=== RESELLER REFERENCE QA ===
Resell points total                    : 2,413
Resell with finite filling_reference   : 2,413
Resell with existing filling_reference : 2,413
Resell with valid reference + time     : 2,413
Missing/invalid reference              : 0
Travel time min/median/mean/p95/max (min): 0.00 / 4.00 / 19.28 / 81.84 / 1087.94
=== RESELLER COST QA ===
Finite cost_res_kg_in                    : 1,832/2,413
Finite cost_res_kg_out                   : 1,8